In [1]:
LOTSA_PATH = r"Datasets\raw_data\LOTSA"
TARGET_PATH = r"Datasets\LOTSA_samples"

In [2]:
import os
import numpy as np
import random
from datasets import load_from_disk
from tqdm import tqdm

# 配置路径
LOTSA_PATH = r"Datasets\raw_data\LOTSA"
TARGET_PATH = r"Datasets\LOTSA_samples"

# 确保输出目录存在
os.makedirs(TARGET_PATH, exist_ok=True)

# 目标参数
TARGET_SAMPLES = 5000
SEQ_LEN = 1000

print(f"源数据路径: {LOTSA_PATH}")
print(f"目标: 从所有合格序列中随机抽取 {TARGET_SAMPLES} 条")

e:\anaconda3\envs\TS_HW2\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


源数据路径: Datasets\raw_data\LOTSA
目标: 从所有合格序列中随机抽取 5000 条


In [3]:
# 蓄水池：用于存放最终选中的样本元数据 [(subset_path, row_index), ...]
reservoir = []
# 全局计数器：记录一共发现了多少条合格的序列
global_valid_count = 0

# 获取所有子集目录
subsets = sorted([d for d in os.listdir(LOTSA_PATH) if os.path.isdir(os.path.join(LOTSA_PATH, d))])

print("开始全局扫描并筛选长序列（蓄水池采样中）...")

for subset_name in tqdm(subsets, desc="扫描子集"):
    subset_path = os.path.join(LOTSA_PATH, subset_name)
    
    try:
        # 加载数据集 (Lazy load)
        ds = load_from_disk(subset_path, keep_in_memory=False)
        if isinstance(ds, dict) and 'train' in ds:
            ds = ds['train']
            
        # 确定数值列名
        target_col = next((col for col in ['target', 'values', 'item_target'] if col in ds.column_names), None)
        if not target_col:
            continue

        # --- 优化点：批量检查长度以提高速度 ---
        # 我们不需要加载整个序列，只需要知道长度。
        # 如果 datasets 版本支持，可以直接访问 column 不加载整个 row。
        # 这里使用 batch 迭代来平衡速度和内存。
        batch_size = 1000
        
        # 遍历数据集
        for i in range(0, len(ds), batch_size):
            # 获取一个 batch 的数据 (仅获取 target 列)
            batch = ds[i : i + batch_size]
            batch_data = batch[target_col]
            
            # 遍历 batch 中的每一条
            for local_idx, ts_seq in enumerate(batch_data):
                # 真实的行号
                real_row_idx = i + local_idx
                
                # 1. 长度检查 (筛选条件)
                # 注意：Arrow读取出来可能是List或Numpy，len()通用
                if len(ts_seq) >= SEQ_LEN:
                    
                    # 2. 蓄水池采样逻辑
                    # 这是一个合格的样本
                    current_k = global_valid_count
                    
                    if current_k < TARGET_SAMPLES:
                        # 如果还没填满 5000 个，直接加入
                        reservoir.append((subset_path, real_row_idx))
                    else:
                        # 如果已经满了，以 k/N 的概率替换掉池子里已有的一个
                        # random.randint(0, current_k) 生成 [0, current_k] 之间的整数
                        # 如果生成的数落在 [0, TARGET_SAMPLES-1] 区间内，则进行替换
                        r = random.randint(0, current_k)
                        if r < TARGET_SAMPLES:
                            reservoir[r] = (subset_path, real_row_idx)
                    
                    global_valid_count += 1

    except Exception as e:
        print(f"  [Error] {subset_name}: {e}")

print(f"\n扫描结束。")
print(f"共发现 {global_valid_count} 条合格序列（长度 >= {SEQ_LEN}）。")
print(f"蓄水池中已保留 {len(reservoir)} 条样本索引。")

开始全局扫描并筛选长序列（蓄水池采样中）...


扫描子集:  77%|███████▋  | 83/108 [06:33<06:02, 14.49s/it]

  [Error] residential_pv_power: 


扫描子集: 100%|██████████| 108/108 [07:28<00:00,  4.15s/it]


扫描结束。
共发现 36369 条合格序列（长度 >= 1000）。
蓄水池中已保留 5000 条样本索引。


In [4]:
from collections import defaultdict

# 1. 重组索引：按文件路径归类，减少文件打开次数
# 格式: { 'path/to/subset': [row_idx1, row_idx2, ...], ... }
tasks = defaultdict(list)
for path, idx in reservoir:
    tasks[path].append(idx)

final_data_list = []

print(f"开始从 {len(tasks)} 个子集中提取实际数据...")

for subset_path, indices in tqdm(tasks.items(), desc="提取数据"):
    try:
        ds = load_from_disk(subset_path, keep_in_memory=False)
        if isinstance(ds, dict) and 'train' in ds:
            ds = ds['train']
            
        target_col = next((col for col in ['target', 'values', 'item_target'] if col in ds.column_names), None)
        
        # 使用 ds.select() 一次性取出该子集所需的所有行，比一个个取快
        # 注意：indices 需要排序
        sorted_indices = sorted(indices)
        subset_selection = ds.select(sorted_indices)
        
        # 获取所有选中行的数值
        all_values = subset_selection[target_col]
        
        # 处理每一条提取出的序列
        for ts_data in all_values:
            ts_np = np.array(ts_data)
            if ts_np.ndim > 1: ts_np = ts_np.flatten()
            
            # 再次确认长度（理论上不需要，但作为双重保险）
            if len(ts_np) >= SEQ_LEN:
                # 随机裁剪 (Random Crop)
                max_start = len(ts_np) - SEQ_LEN
                start = random.randint(0, max_start)
                crop = ts_np[start : start + SEQ_LEN]
                
                final_data_list.append(crop)
                
    except Exception as e:
        print(f"提取失败 {subset_path}: {e}")

# 转换为 Numpy 数组
final_data = np.array(final_data_list)
print(f"\n提取完成。最终数据形状: {final_data.shape}")

开始从 47 个子集中提取实际数据...


提取数据: 100%|██████████| 47/47 [00:44<00:00,  1.05it/s]


提取完成。最终数据形状: (5000, 1000)


In [5]:
if final_data.shape[0] > 0:
    # 再次打乱，消除按文件夹读取带来的顺序性
    np.random.shuffle(final_data)
    
    output_file = os.path.join(TARGET_PATH, "lotsa_verified_sample_5k.npy")
    np.save(output_file, final_data)
    
    print(f"成功保存至: {output_file}")
    print(f"数据统计 - Mean: {np.mean(final_data):.4f}, Std: {np.std(final_data):.4f}")
    
    # 如果样本不足 5000 (极为罕见的情况，只有当总合格数 < 5000 时发生)
    if final_data.shape[0] < TARGET_SAMPLES:
        print(f"注意：整个数据集中仅发现 {final_data.shape[0]} 条合格长序列。")
else:
    print("错误：未生成任何数据。")

成功保存至: Datasets\LOTSA_samples\lotsa_verified_sample_5k.npy
数据统计 - Mean: nan, Std: nan


In [6]:
#检查形状，打印元素
print(final_data.shape)

(5000, 1000)


In [7]:
print(final_data)

[[1.10000000e+01 7.00000000e+00 7.00000000e+00 ... 3.60000000e+01
  4.40000000e+01 5.90000000e+01]
 [1.14000000e-01 1.12000003e-01 1.11000001e-01 ... 9.01000023e-01
  5.74000001e-01 5.50999999e-01]
 [1.11500002e-01 1.11500002e-01 1.11000001e-01 ... 8.94999951e-02
  1.91000000e-01 1.00000001e-01]
 ...
 [3.46000000e+02 3.70000000e+02 3.80000000e+02 ... 6.50000000e+01
  5.80000000e+01 6.00000000e+01]
 [2.40000000e+01 1.40000000e+01 2.30000000e+01 ... 6.00000000e+00
  1.00000000e+01 1.20000000e+01]
 [2.23000000e+02 2.63000000e+02 2.70000000e+02 ... 3.08000000e+02
  4.44000000e+02 4.06000000e+02]]
